In [1]:
from pathlib import Path
from pynq import Overlay

print("Working directory:", Path.cwd())

# Parse the hardware metadata without programming the FPGA.
overlay = Overlay("axis_dummy_overlay.bit", download=False)

for name, info in overlay.ip_dict.items():
    print(name, info.get("type"))

Working directory: /home/root/jupyter_notebooks/elen90096/1647718/Project


axi_dma_0 xilinx.com:ip:axi_dma:7.1
axis_dummy_top_0 xilinx.com:hls:axis_dummy_top:1.0
zynq_ultra_ps_e_0 xilinx.com:ip:zynq_ultra_ps_e:3.5


In [2]:
overlay.download()

dma = overlay.axi_dma_0
dummy = overlay.axis_dummy_top_0

print("Overlay loaded:", overlay.is_loaded())
print("DMA maximum transfer size (bytes):", dma.buffer_max_size)
print("Dummy control register:", hex(dummy.read(0x00)))
print("DMA MM2S status:", hex(dma.read(0x04)))
print("DMA S2MM status:", hex(dma.read(0x34)))

Overlay loaded: True
DMA maximum transfer size (bytes): 16383
Dummy control register: 0x4
DMA MM2S status: 0x0
DMA S2MM status: 0x0


In [3]:
import numpy as np
import time
from pynq import allocate

src = allocate(shape=(5,), dtype=np.uint32)
dst = allocate(shape=(5,), dtype=np.uint32)

src[:] = [0, 1, 42, 123, 0xFFFFFFFF]
dst[:] = 0
src.flush()
dst.flush()

# Arm the receive channel before starting the kernel and transmission.
dma.recvchannel.transfer(dst)
dummy.write(0x10, 5)
dummy.write(0x00, 1)
dma.sendchannel.transfer(src)

deadline = time.monotonic() + 5

while not (dma.sendchannel.idle and dma.recvchannel.idle):
    if time.monotonic() > deadline:
        raise TimeoutError(
            f"DMA timeout: MM2S={hex(dma.read(0x04))}, "
            f"S2MM={hex(dma.read(0x34))}"
        )
    time.sleep(0.001)

dma.sendchannel.wait()
dma.recvchannel.wait()
dst.invalidate()

print("Input:", np.array(src))
print("Output:", np.array(dst))

expected = np.array([1, 2, 43, 124, 0], dtype=np.uint32)
assert np.array_equal(dst, expected), "Output mismatch"

print("PASS: Initial on-board DMA + dummy correctness test")

src.freebuffer()
dst.freebuffer()

Input: [         0          1         42        123 4294967295]
Output: [  1   2  43 124   0]
PASS: Initial on-board DMA + dummy correctness test


In [4]:
%run axis_dummy_p0c.py --bit axis_dummy_overlay.bit --lengths 1 5 257 4095 --iterations 100 --reloads 3 --log axis_dummy_p0c_log.json

PASS reload=0 iteration=0 length=1
PASS reload=0 iteration=0 length=5
PASS reload=0 iteration=0 length=257
PASS reload=0 iteration=0 length=4095
PASS reload=0 iteration=1 length=1
PASS reload=0 iteration=1 length=5
PASS reload=0 iteration=1 length=257
PASS reload=0 iteration=1 length=4095
PASS reload=0 iteration=2 length=1
PASS reload=0 iteration=2 length=5
PASS reload=0 iteration=2 length=257
PASS reload=0 iteration=2 length=4095
PASS reload=0 iteration=3 length=1
PASS reload=0 iteration=3 length=5
PASS reload=0 iteration=3 length=257
PASS reload=0 iteration=3 length=4095
PASS reload=0 iteration=4 length=1
PASS reload=0 iteration=4 length=5
PASS reload=0 iteration=4 length=257
PASS reload=0 iteration=4 length=4095
PASS reload=0 iteration=5 length=1
PASS reload=0 iteration=5 length=5
PASS reload=0 iteration=5 length=257
PASS reload=0 iteration=5 length=4095
PASS reload=0 iteration=6 length=1
PASS reload=0 iteration=6 length=5
PASS reload=0 iteration=6 length=257
PASS reload=0 iteration

PASS reload=0 iteration=81 length=4095
PASS reload=0 iteration=82 length=1
PASS reload=0 iteration=82 length=5
PASS reload=0 iteration=82 length=257
PASS reload=0 iteration=82 length=4095
PASS reload=0 iteration=83 length=1
PASS reload=0 iteration=83 length=5
PASS reload=0 iteration=83 length=257
PASS reload=0 iteration=83 length=4095
PASS reload=0 iteration=84 length=1
PASS reload=0 iteration=84 length=5
PASS reload=0 iteration=84 length=257
PASS reload=0 iteration=84 length=4095
PASS reload=0 iteration=85 length=1
PASS reload=0 iteration=85 length=5
PASS reload=0 iteration=85 length=257
PASS reload=0 iteration=85 length=4095
PASS reload=0 iteration=86 length=1
PASS reload=0 iteration=86 length=5
PASS reload=0 iteration=86 length=257
PASS reload=0 iteration=86 length=4095
PASS reload=0 iteration=87 length=1
PASS reload=0 iteration=87 length=5
PASS reload=0 iteration=87 length=257
PASS reload=0 iteration=87 length=4095
PASS reload=0 iteration=88 length=1
PASS reload=0 iteration=88 leng

PASS reload=1 iteration=54 length=5
PASS reload=1 iteration=54 length=257
PASS reload=1 iteration=54 length=4095
PASS reload=1 iteration=55 length=1
PASS reload=1 iteration=55 length=5
PASS reload=1 iteration=55 length=257
PASS reload=1 iteration=55 length=4095
PASS reload=1 iteration=56 length=1
PASS reload=1 iteration=56 length=5
PASS reload=1 iteration=56 length=257
PASS reload=1 iteration=56 length=4095
PASS reload=1 iteration=57 length=1
PASS reload=1 iteration=57 length=5
PASS reload=1 iteration=57 length=257
PASS reload=1 iteration=57 length=4095
PASS reload=1 iteration=58 length=1
PASS reload=1 iteration=58 length=5
PASS reload=1 iteration=58 length=257
PASS reload=1 iteration=58 length=4095
PASS reload=1 iteration=59 length=1
PASS reload=1 iteration=59 length=5
PASS reload=1 iteration=59 length=257
PASS reload=1 iteration=59 length=4095
PASS reload=1 iteration=60 length=1
PASS reload=1 iteration=60 length=5
PASS reload=1 iteration=60 length=257
PASS reload=1 iteration=60 lengt

PASS reload=2 iteration=27 length=5
PASS reload=2 iteration=27 length=257
PASS reload=2 iteration=27 length=4095
PASS reload=2 iteration=28 length=1
PASS reload=2 iteration=28 length=5
PASS reload=2 iteration=28 length=257
PASS reload=2 iteration=28 length=4095
PASS reload=2 iteration=29 length=1
PASS reload=2 iteration=29 length=5
PASS reload=2 iteration=29 length=257
PASS reload=2 iteration=29 length=4095
PASS reload=2 iteration=30 length=1
PASS reload=2 iteration=30 length=5
PASS reload=2 iteration=30 length=257
PASS reload=2 iteration=30 length=4095
PASS reload=2 iteration=31 length=1
PASS reload=2 iteration=31 length=5
PASS reload=2 iteration=31 length=257
PASS reload=2 iteration=31 length=4095
PASS reload=2 iteration=32 length=1
PASS reload=2 iteration=32 length=5
PASS reload=2 iteration=32 length=257
PASS reload=2 iteration=32 length=4095
PASS reload=2 iteration=33 length=1
PASS reload=2 iteration=33 length=5
PASS reload=2 iteration=33 length=257
PASS reload=2 iteration=33 lengt

In [5]:
import json
from pathlib import Path

log_path = Path("axis_dummy_p0c_log.json")

with log_path.open(encoding="utf-8") as f:
    test_log = json.load(f)

print("Log file:", log_path.resolve())
print("Top-level fields:", list(test_log))
print(json.dumps(test_log, indent=2)[:4000])

Log file: /home/root/jupyter_notebooks/elen90096/1647718/Project/axis_dummy_p0c_log.json
Top-level fields: ['status', 'bitstream', 'lengths', 'iterations', 'reloads', 'elapsed_seconds', 'results']
{
  "status": "PASS",
  "bitstream": "/home/root/jupyter_notebooks/elen90096/1647718/Project/axis_dummy_overlay.bit",
  "lengths": [
    1,
    5,
    257,
    4095
  ],
  "iterations": 100,
  "reloads": 3,
  "elapsed_seconds": 4.772479421999833,
  "results": [
    {
      "length": 1,
      "iteration": 0,
      "status": "PASS",
      "overlay_reload": 0
    },
    {
      "length": 5,
      "iteration": 0,
      "status": "PASS",
      "overlay_reload": 0
    },
    {
      "length": 257,
      "iteration": 0,
      "status": "PASS",
      "overlay_reload": 0
    },
    {
      "length": 4095,
      "iteration": 0,
      "status": "PASS",
      "overlay_reload": 0
    },
    {
      "length": 1,
      "iteration": 1,
      "status": "PASS",
      "overlay_reload": 0
    },
    {
      "len

In [6]:
from collections import Counter

# Find the list of per-transfer records.
records = next(
    value for value in test_log.values()
    if isinstance(value, list)
    and value
    and isinstance(value[0], dict)
    and "overlay_reload" in value[0]
)

status_counts = Counter(r["status"] for r in records)
length_counts = Counter(r["length"] for r in records)
reload_counts = Counter(r["overlay_reload"] for r in records)

print("=== On-board DMA + Dummy Validation ===")
print("Total transfers:", len(records))
print("Status counts:", dict(status_counts))
print("Transfers per length:", dict(sorted(length_counts.items())))
print("Transfers per overlay load:", dict(sorted(reload_counts.items())))

expected_cases = {
    (reload_id, iteration, length)
    for reload_id in range(3)
    for iteration in range(100)
    for length in (1, 5, 257, 4095)
}
actual_cases = {
    (r["overlay_reload"], r["iteration"], r["length"])
    for r in records
}

assert len(records) == 1200, "Unexpected transfer count"
assert actual_cases == expected_cases, "Missing or unexpected test cases"
assert all(r["status"] == "PASS" for r in records), "Failed test detected"

print("PASS: All 1,200 transfers verified across 3 overlay loads.")

=== On-board DMA + Dummy Validation ===
Total transfers: 1200
Status counts: {'PASS': 1200}
Transfers per length: {1: 300, 5: 300, 257: 300, 4095: 300}
Transfers per overlay load: {0: 400, 1: 400, 2: 400}
PASS: All 1,200 transfers verified across 3 overlay loads.


In [7]:
import json
import time
import numpy as np
from pynq import Overlay, allocate
from pathlib import Path

overlay = Overlay("axis_dummy_overlay.bit")
dma = overlay.axi_dma_0
dummy = overlay.axis_dummy_top_0

benchmark_results = []

print("=== DMA + Dummy Host-Observed Benchmark ===")
print("Warm-up: 10 transfers; measured: 100 transfers per length")
print("Throughput counts input payload only (decimal MB/s).")
print()

for length in (1, 5, 257, 4095):
    assert length * 4 <= dma.buffer_max_size, "DMA transfer too large"

    src = allocate(shape=(length,), dtype=np.uint32)
    dst = allocate(shape=(length,), dtype=np.uint32)
    src[:] = np.arange(length, dtype=np.uint32)
    expected = np.arange(length, dtype=np.uint32) + np.uint32(1)
    src.flush()

    samples_us = []

    for iteration in range(110):
        dst[:] = 0
        dst.flush()

        start_ns = time.perf_counter_ns()

        dma.recvchannel.transfer(dst)
        dummy.write(0x10, length)
        dummy.write(0x00, 1)
        dma.sendchannel.transfer(src)

        deadline = time.monotonic() + 5
        while not (dma.sendchannel.idle and dma.recvchannel.idle):
            if time.monotonic() > deadline:
                raise TimeoutError(
                    f"DMA timeout: MM2S={hex(dma.read(0x04))}, "
                    f"S2MM={hex(dma.read(0x34))}"
                )

        dma.sendchannel.wait()
        dma.recvchannel.wait()
        elapsed_us = (time.perf_counter_ns() - start_ns) / 1000

        dst.invalidate()
        assert np.array_equal(dst, expected), (
            f"Output mismatch: length={length}, iteration={iteration}"
        )

        if iteration >= 10:
            samples_us.append(elapsed_us)

    median_us = float(np.median(samples_us))
    result = {
        "length_words": length,
        "payload_bytes": length * 4,
        "measured_transfers": len(samples_us),
        "median_latency_us": median_us,
        "p95_latency_us": float(np.percentile(samples_us, 95)),
        "input_payload_MB_per_s": length * 4 / median_us,
        "latency_samples_us": samples_us,
        "status": "PASS",
    }
    benchmark_results.append(result)

    print(
        f"Length={length:4d} | "
        f"Median={median_us:.2f} us | "
        f"P95={result['p95_latency_us']:.2f} us | "
        f"Input throughput={result['input_payload_MB_per_s']:.2f} MB/s"
    )

    src.freebuffer()
    dst.freebuffer()

report = {
    "measurement": "Host-observed DMA + dummy transaction latency",
    "throughput_definition": "Input payload bytes / transaction time",
    "warmup_transfers_per_length": 10,
    "results": benchmark_results,
}

output_path = Path("axis_dummy_benchmark.json")
output_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print()
print("PASS: All benchmark outputs verified.")
print("Saved:", output_path.resolve())

=== DMA + Dummy Host-Observed Benchmark ===
Warm-up: 10 transfers; measured: 100 transfers per length
Throughput counts input payload only (decimal MB/s).

Length=   1 | Median=362.37 us | P95=378.90 us | Input throughput=0.01 MB/s
Length=   5 | Median=361.89 us | P95=377.40 us | Input throughput=0.06 MB/s
Length= 257 | Median=362.53 us | P95=377.96 us | Input throughput=2.84 MB/s
Length=4095 | Median=388.77 us | P95=402.98 us | Input throughput=42.13 MB/s

PASS: All benchmark outputs verified.
Saved: /home/root/jupyter_notebooks/elen90096/1647718/Project/axis_dummy_benchmark.json


In [1]:
from pynq import Overlay

overlay = Overlay("axis_dummy_overlay_btt26.bit")
dma = overlay.axi_dma_0
dummy = overlay.axis_dummy_top_0

print("Overlay loaded:", overlay.is_loaded())
print("DMA maximum transfer size (bytes):", dma.buffer_max_size)

assert dma.buffer_max_size == (1 << 26) - 1, (
    "Unexpected DMA buffer length configuration"
)
print("PASS: 26-bit DMA transfer limit confirmed.")

Overlay loaded: True
DMA maximum transfer size (bytes): 67108863
PASS: 26-bit DMA transfer limit confirmed.


In [2]:
import time
import numpy as np
from pynq import allocate

length = 65025
src = allocate(shape=(length,), dtype=np.uint32)
dst = allocate(shape=(length,), dtype=np.uint32)

src[:] = np.arange(length, dtype=np.uint32)
src[0] = 0xFFFFFFFF
expected = np.array(src) + np.uint32(1)

dst[:] = 0
src.flush()
dst.flush()

# Arm reception before starting the kernel and transmission.
dma.recvchannel.transfer(dst)
dummy.write(0x10, length)
dummy.write(0x00, 1)
dma.sendchannel.transfer(src)

deadline = time.monotonic() + 5

while not (dma.sendchannel.idle and dma.recvchannel.idle):
    if time.monotonic() > deadline:
        raise TimeoutError(
            f"DMA timeout: MM2S={hex(dma.read(0x04))}, "
            f"S2MM={hex(dma.read(0x34))}"
        )
    time.sleep(0.001)

dma.sendchannel.wait()
dma.recvchannel.wait()
dst.invalidate()

assert np.array_equal(dst, expected), "Output mismatch"

print("Transfer length (words):", length)
print("Input payload (bytes):", src.nbytes)
print("First five outputs:", np.array(dst[:5]))
print("Last five outputs:", np.array(dst[-5:]))
print("PASS: All 65,025 output words match the expected result.")

src.freebuffer()
dst.freebuffer()

Transfer length (words): 65025
Input payload (bytes): 260100
First five outputs: [0 2 3 4 5]
Last five outputs: [65021 65022 65023 65024 65025]
PASS: All 65,025 output words match the expected result.


In [3]:
%run axis_dummy_p0c.py --bit axis_dummy_overlay_btt26.bit --lengths 1 5 257 4095 65025 --iterations 100 --reloads 3 --log axis_dummy_p0c_btt26_log.json

PASS reload=0 iteration=0 length=1
PASS reload=0 iteration=0 length=5
PASS reload=0 iteration=0 length=257
PASS reload=0 iteration=0 length=4095
PASS reload=0 iteration=0 length=65025
PASS reload=0 iteration=1 length=1
PASS reload=0 iteration=1 length=5
PASS reload=0 iteration=1 length=257
PASS reload=0 iteration=1 length=4095
PASS reload=0 iteration=1 length=65025
PASS reload=0 iteration=2 length=1
PASS reload=0 iteration=2 length=5
PASS reload=0 iteration=2 length=257
PASS reload=0 iteration=2 length=4095
PASS reload=0 iteration=2 length=65025
PASS reload=0 iteration=3 length=1
PASS reload=0 iteration=3 length=5
PASS reload=0 iteration=3 length=257
PASS reload=0 iteration=3 length=4095
PASS reload=0 iteration=3 length=65025
PASS reload=0 iteration=4 length=1
PASS reload=0 iteration=4 length=5
PASS reload=0 iteration=4 length=257
PASS reload=0 iteration=4 length=4095
PASS reload=0 iteration=4 length=65025
PASS reload=0 iteration=5 length=1
PASS reload=0 iteration=5 length=5
PASS reloa

PASS reload=0 iteration=57 length=65025
PASS reload=0 iteration=58 length=1
PASS reload=0 iteration=58 length=5
PASS reload=0 iteration=58 length=257
PASS reload=0 iteration=58 length=4095
PASS reload=0 iteration=58 length=65025
PASS reload=0 iteration=59 length=1
PASS reload=0 iteration=59 length=5
PASS reload=0 iteration=59 length=257
PASS reload=0 iteration=59 length=4095
PASS reload=0 iteration=59 length=65025
PASS reload=0 iteration=60 length=1
PASS reload=0 iteration=60 length=5
PASS reload=0 iteration=60 length=257
PASS reload=0 iteration=60 length=4095
PASS reload=0 iteration=60 length=65025
PASS reload=0 iteration=61 length=1
PASS reload=0 iteration=61 length=5
PASS reload=0 iteration=61 length=257
PASS reload=0 iteration=61 length=4095
PASS reload=0 iteration=61 length=65025
PASS reload=0 iteration=62 length=1
PASS reload=0 iteration=62 length=5
PASS reload=0 iteration=62 length=257
PASS reload=0 iteration=62 length=4095
PASS reload=0 iteration=62 length=65025
PASS reload=0 i

PASS reload=1 iteration=14 length=65025
PASS reload=1 iteration=15 length=1
PASS reload=1 iteration=15 length=5
PASS reload=1 iteration=15 length=257
PASS reload=1 iteration=15 length=4095
PASS reload=1 iteration=15 length=65025
PASS reload=1 iteration=16 length=1
PASS reload=1 iteration=16 length=5
PASS reload=1 iteration=16 length=257
PASS reload=1 iteration=16 length=4095
PASS reload=1 iteration=16 length=65025
PASS reload=1 iteration=17 length=1
PASS reload=1 iteration=17 length=5
PASS reload=1 iteration=17 length=257
PASS reload=1 iteration=17 length=4095
PASS reload=1 iteration=17 length=65025
PASS reload=1 iteration=18 length=1
PASS reload=1 iteration=18 length=5
PASS reload=1 iteration=18 length=257
PASS reload=1 iteration=18 length=4095
PASS reload=1 iteration=18 length=65025
PASS reload=1 iteration=19 length=1
PASS reload=1 iteration=19 length=5
PASS reload=1 iteration=19 length=257
PASS reload=1 iteration=19 length=4095
PASS reload=1 iteration=19 length=65025
PASS reload=1 i

PASS reload=1 iteration=58 length=4095
PASS reload=1 iteration=58 length=65025
PASS reload=1 iteration=59 length=1
PASS reload=1 iteration=59 length=5
PASS reload=1 iteration=59 length=257
PASS reload=1 iteration=59 length=4095
PASS reload=1 iteration=59 length=65025
PASS reload=1 iteration=60 length=1
PASS reload=1 iteration=60 length=5
PASS reload=1 iteration=60 length=257
PASS reload=1 iteration=60 length=4095
PASS reload=1 iteration=60 length=65025
PASS reload=1 iteration=61 length=1
PASS reload=1 iteration=61 length=5
PASS reload=1 iteration=61 length=257
PASS reload=1 iteration=61 length=4095
PASS reload=1 iteration=61 length=65025
PASS reload=1 iteration=62 length=1
PASS reload=1 iteration=62 length=5
PASS reload=1 iteration=62 length=257
PASS reload=1 iteration=62 length=4095
PASS reload=1 iteration=62 length=65025
PASS reload=1 iteration=63 length=1
PASS reload=1 iteration=63 length=5
PASS reload=1 iteration=63 length=257
PASS reload=1 iteration=63 length=4095
PASS reload=1 it

PASS reload=2 iteration=14 length=65025
PASS reload=2 iteration=15 length=1
PASS reload=2 iteration=15 length=5
PASS reload=2 iteration=15 length=257
PASS reload=2 iteration=15 length=4095
PASS reload=2 iteration=15 length=65025
PASS reload=2 iteration=16 length=1
PASS reload=2 iteration=16 length=5
PASS reload=2 iteration=16 length=257
PASS reload=2 iteration=16 length=4095
PASS reload=2 iteration=16 length=65025
PASS reload=2 iteration=17 length=1
PASS reload=2 iteration=17 length=5
PASS reload=2 iteration=17 length=257
PASS reload=2 iteration=17 length=4095
PASS reload=2 iteration=17 length=65025
PASS reload=2 iteration=18 length=1
PASS reload=2 iteration=18 length=5
PASS reload=2 iteration=18 length=257
PASS reload=2 iteration=18 length=4095
PASS reload=2 iteration=18 length=65025
PASS reload=2 iteration=19 length=1
PASS reload=2 iteration=19 length=5
PASS reload=2 iteration=19 length=257
PASS reload=2 iteration=19 length=4095
PASS reload=2 iteration=19 length=65025
PASS reload=2 i

PASS reload=2 iteration=72 length=4095
PASS reload=2 iteration=72 length=65025
PASS reload=2 iteration=73 length=1
PASS reload=2 iteration=73 length=5
PASS reload=2 iteration=73 length=257
PASS reload=2 iteration=73 length=4095
PASS reload=2 iteration=73 length=65025
PASS reload=2 iteration=74 length=1
PASS reload=2 iteration=74 length=5
PASS reload=2 iteration=74 length=257
PASS reload=2 iteration=74 length=4095
PASS reload=2 iteration=74 length=65025
PASS reload=2 iteration=75 length=1
PASS reload=2 iteration=75 length=5
PASS reload=2 iteration=75 length=257
PASS reload=2 iteration=75 length=4095
PASS reload=2 iteration=75 length=65025
PASS reload=2 iteration=76 length=1
PASS reload=2 iteration=76 length=5
PASS reload=2 iteration=76 length=257
PASS reload=2 iteration=76 length=4095
PASS reload=2 iteration=76 length=65025
PASS reload=2 iteration=77 length=1
PASS reload=2 iteration=77 length=5
PASS reload=2 iteration=77 length=257
PASS reload=2 iteration=77 length=4095
PASS reload=2 it

In [4]:
import json
from pathlib import Path
from collections import Counter

log_path = Path("axis_dummy_p0c_btt26_log.json")
test_log = json.loads(log_path.read_text(encoding="utf-8"))
records = test_log["results"]

expected_cases = {
    (reload_id, iteration, length)
    for reload_id in range(3)
    for iteration in range(100)
    for length in (1, 5, 257, 4095, 65025)
}
actual_cases = {
    (r["overlay_reload"], r["iteration"], r["length"])
    for r in records
}

assert len(records) == 1500, "Unexpected transfer count"
assert actual_cases == expected_cases, "Missing or unexpected test cases"
assert all(r["status"] == "PASS" for r in records), "Failed test detected"

print("=== 26-bit DMA + Dummy Validation ===")
print("Total transfers:", len(records))
print("Status counts:", dict(Counter(r["status"] for r in records)))
print("Transfers per length:",
      dict(sorted(Counter(r["length"] for r in records).items())))
print("Transfers per overlay load:",
      dict(sorted(Counter(r["overlay_reload"] for r in records).items())))
print("PASS: All 1,500 unique test cases verified.")

=== 26-bit DMA + Dummy Validation ===
Total transfers: 1500
Status counts: {'PASS': 1500}
Transfers per length: {1: 300, 5: 300, 257: 300, 4095: 300, 65025: 300}
Transfers per overlay load: {0: 500, 1: 500, 2: 500}
PASS: All 1,500 unique test cases verified.


In [5]:
import json
import time
import numpy as np
from pynq import Overlay, allocate
from pathlib import Path

overlay = Overlay("axis_dummy_overlay_btt26.bit")
dma = overlay.axi_dma_0
dummy = overlay.axis_dummy_top_0

assert dma.buffer_max_size == (1 << 26) - 1, (
    "Unexpected DMA buffer length configuration"
)

benchmark_results = []

print("=== 26-bit DMA + Dummy Host-Observed Benchmark ===")
print("Warm-up: 10 transfers; measured: 100 transfers per length")
print("Throughput counts input payload only (decimal MB/s).")
print("Timing includes Python/DMA control and completion polling.")
print()

for length in (1, 5, 257, 4095, 65025):
    assert length * 4 <= dma.buffer_max_size, "DMA transfer too large"

    src = allocate(shape=(length,), dtype=np.uint32)
    dst = allocate(shape=(length,), dtype=np.uint32)

    src[:] = np.arange(length, dtype=np.uint32)
    expected = np.arange(length, dtype=np.uint32) + np.uint32(1)
    src.flush()

    samples_us = []

    for iteration in range(110):
        dst[:] = 0
        dst.flush()

        start_ns = time.perf_counter_ns()

        # Arm reception before starting the kernel and transmission.
        dma.recvchannel.transfer(dst)
        dummy.write(0x10, length)
        dummy.write(0x00, 1)
        dma.sendchannel.transfer(src)

        deadline = time.monotonic() + 5

        while not (dma.sendchannel.idle and dma.recvchannel.idle):
            if time.monotonic() > deadline:
                raise TimeoutError(
                    f"DMA timeout: MM2S={hex(dma.read(0x04))}, "
                    f"S2MM={hex(dma.read(0x34))}"
                )

        dma.sendchannel.wait()
        dma.recvchannel.wait()

        elapsed_us = (time.perf_counter_ns() - start_ns) / 1000

        # Validate the output outside the timed interval.
        dst.invalidate()
        assert np.array_equal(dst, expected), (
            f"Output mismatch: length={length}, iteration={iteration}"
        )

        if iteration >= 10:
            samples_us.append(elapsed_us)

    median_us = float(np.median(samples_us))

    result = {
        "length_words": length,
        "payload_bytes": length * 4,
        "measured_transfers": len(samples_us),
        "median_latency_us": median_us,
        "p95_latency_us": float(np.percentile(samples_us, 95)),
        "input_payload_MB_per_s": length * 4 / median_us,
        "latency_samples_us": samples_us,
        "status": "PASS",
    }
    benchmark_results.append(result)

    print(
        f"Length={length:5d} | "
        f"Median={median_us:.2f} us | "
        f"P95={result['p95_latency_us']:.2f} us | "
        f"Input throughput={result['input_payload_MB_per_s']:.2f} MB/s"
    )

    src.freebuffer()
    dst.freebuffer()

report = {
    "bitstream": str(Path("axis_dummy_overlay_btt26.bit").resolve()),
    "dma_max_transfer_bytes": dma.buffer_max_size,
    "measurement": "Host-observed DMA + dummy transaction latency",
    "throughput_definition": "Input payload bytes / transaction time",
    "warmup_transfers_per_length": 10,
    "results": benchmark_results,
}

output_path = Path("axis_dummy_benchmark_btt26.json")
output_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print()
print("PASS: All benchmark outputs verified.")
print("Saved:", output_path.resolve())

=== 26-bit DMA + Dummy Host-Observed Benchmark ===
Warm-up: 10 transfers; measured: 100 transfers per length
Throughput counts input payload only (decimal MB/s).
Timing includes Python/DMA control and completion polling.

Length=    1 | Median=357.81 us | P95=372.07 us | Input throughput=0.01 MB/s
Length=    5 | Median=356.74 us | P95=371.13 us | Input throughput=0.06 MB/s
Length=  257 | Median=359.65 us | P95=374.16 us | Input throughput=2.86 MB/s
Length= 4095 | Median=386.21 us | P95=402.36 us | Input throughput=42.41 MB/s
Length=65025 | Median=885.97 us | P95=941.39 us | Input throughput=293.58 MB/s

PASS: All benchmark outputs verified.
Saved: /home/root/jupyter_notebooks/elen90096/1647718/Project/axis_dummy_benchmark_btt26.json
